In [1]:
import pandas as pd
import re
import nltk
from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [2]:
df_sampled = pd.read_csv('training_data_papers.csv')

In [3]:
# Download standard stopwords if you haven't already
nltk.download('stopwords')

# 1. Setup our tools
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

# 2. Add Domain-Specific Stop Words!
# These words appear in almost every ArXiv paper and carry zero semantic meaning.
# If you don't remove them, they will dominate your SVD dimensions.
domain_stops = {"paper", "propose", "method", "algorithm", "show", "results", 
                "we", "use", "approach", "model", "two", "one", "new"}
stop_words = stop_words.union(domain_stops)

def tokenize_and_clean(text):
    # Lowercase everything
    text = text.lower()
    
    # Remove punctuation, numbers, and special characters (keep only a-z)
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # Split into words, remove stop words, remove tiny words, and stem
    tokens = []
    for word in text.split():
        if word not in stop_words and len(word) > 2:
            tokens.append(stemmer.stem(word))
            
    return tokens

print("Tokenizing and stemming all papers...")
# Assuming df_sampled has the 'abstract_clean' column from our LaTeX removal step
# This creates a list of lists: [["novel", "graph", "travers"], ["deep", "learn", "vision"], ...]
documents_tokens = df_sampled['abstract_clean'].apply(tokenize_and_clean).tolist()

[nltk_data] Downloading package stopwords to /home/bam__/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Tokenizing and stemming all papers...


In [5]:
print("Calculating Document Frequencies...")
doc_freq = Counter()

# Count how many documents each word appears in
for tokens in documents_tokens:
    # Use set() because we only care IF it appears in the doc, not how many times
    unique_words_in_doc = set(tokens) 
    doc_freq.update(unique_words_in_doc)

total_docs = len(documents_tokens)
min_df_threshold = 10                  # Must appear in at least 10 papers
max_df_threshold = 0.80 * total_docs   # Must not appear in more than 80% of papers

# Build the final allowed vocabulary
final_vocab = {}
vocab_index = 0

for word, freq in doc_freq.items():
    if min_df_threshold <= freq <= max_df_threshold:
        final_vocab[word] = vocab_index
        vocab_index += 1

print(f"Original unique words: {len(doc_freq)}")
print(f"Pruned Vocabulary Size: {len(final_vocab)} words!")

Calculating Document Frequencies...
Original unique words: 225869
Pruned Vocabulary Size: 29024 words!


In [7]:
print("Re-building documents_integer_ids...")
documents_integer_ids = []

# Swap the string tokens for integer IDs based on our pruned vocab
for tokens in documents_tokens:
    doc_ids = [final_vocab[word] for word in tokens if word in final_vocab]
    documents_integer_ids.append(doc_ids)
    
print(f"Successfully built {len(documents_integer_ids)} integer arrays!")

Re-building documents_integer_ids...
Successfully built 499998 integer arrays!


In [8]:
import json
import pandas as pd

print("Saving checkpoint data for C++ backend...")

# 1. Save the Vocabulary (The Decoder Ring)
with open('vocab_mapping.json', 'w') as f:
    json.dump(final_vocab, f)
print("Saved vocab_mapping.json")

# 2. Save the Document Integer IDs (The Math Data)
# A plain text file where each line is one document, and numbers are separated by spaces.
# C++ std::ifstream can parse this millions of times faster than parsing JSON.
with open('documents_integers.txt', 'w') as f:
    for doc in documents_integer_ids:
        # Convert list of ints to a space-separated string: "45 102 9 12"
        line = " ".join(map(str, doc))
        f.write(line + "\n")
print("Saved documents_integers.txt")

# 3. Save the Original Papers (The Frontend Display)
# We only need the ID, original title, and original abstract to show the user later.
df_frontend = df_sampled[['id', 'title', 'abstract']]
df_frontend.to_csv('frontend_papers.csv', index=False)
print("Saved frontend_papers.csv")

print("Checkpoint complete! You can safely close Python.")

Saving checkpoint data for C++ backend...
Saved vocab_mapping.json
Saved documents_integers.txt
Saved frontend_papers.csv
Checkpoint complete! You can safely close Python.
